# 06 - Cleaning CIFs with SAMOSA

This notebook converts CIFs to P1 symmetry and then runs SAMOSA (`main.py`) in batch mode.

It is configured for your folder:
`C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\CIFs_to_modify\To_modify`

Workflow:
1. Validate environment and clone/update SAMOSA.
2. Convert all input CIFs to P1 using `scripts/clean_csd_mofs.py`.
3. Run SAMOSA on all converted CIFs.
4. Summarize outputs and quality-control flags.

## Prerequisites

- CSD Python API (`ccdc`) must be installed and licensed.
- `git` must be available on PATH.
- Internet is needed the first time to clone SAMOSA.
- Pip-installable dependencies used here: `pandas`, `pymatgen`, `mendeleev`.

SAMOSA repository used in this notebook:
https://github.com/uowoolab/SAMOSA

In [1]:
from pathlib import Path
from datetime import datetime
import importlib
import os
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import display

# -------------------------
# User configuration
# -------------------------
INPUT_CIF_DIR = Path(r"C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\CIFs_to_modify\To_modify_MS")

PROJECT_ROOT = Path.cwd()
SAMOSA_REPO_DIR = PROJECT_ROOT / "external" / "SAMOSA"
P1_CONVERTER_SCRIPT = PROJECT_ROOT / "scripts" / "clean_csd_mofs.py"

PIPELINE_ROOT = PROJECT_ROOT / "data" / "samosa_pipeline"
P1_OUTPUT_DIR = PIPELINE_ROOT / "01_p1_cifs_MS"
SAMOSA_OUTPUT_DIR = PIPELINE_ROOT / "02_samosa_output"
REPORTS_DIR = PIPELINE_ROOT / "reports"

# Runtime controls
AUTO_INSTALL_PY_DEPS = True
OVERWRITE_P1 = False
OVERWRITE_SAMOSA_OUTPUT = False

N_PROCESSES = max(1, min(8, (os.cpu_count() or 4) - 1))
REMOVABLE_DENTICITY = 1
KEEP_BOUND = False
KEEP_OXO = False
VERBOSE = True
LOGGING_LEVEL = "INFO"

In [2]:
def run_cmd(cmd, cwd=None, check=True, show_stdout=True, show_stderr=True):
    """Run a subprocess and return CompletedProcess while printing command and output."""
    cmd = [str(x) for x in cmd]
    print(f"\n$ {' '.join(cmd)}")
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True,
    )
    if show_stdout and result.stdout.strip():
        print(result.stdout.strip())
    if show_stderr and result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(cmd)}")
    return result


for path in [PIPELINE_ROOT, P1_OUTPUT_DIR, SAMOSA_OUTPUT_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Input CIF directory: {INPUT_CIF_DIR}")
print(f"P1 converter script: {P1_CONVERTER_SCRIPT}")
print(f"SAMOSA repo directory: {SAMOSA_REPO_DIR}")

if not INPUT_CIF_DIR.exists():
    raise FileNotFoundError(f"Input directory does not exist: {INPUT_CIF_DIR}")

if not P1_CONVERTER_SCRIPT.exists():
    raise FileNotFoundError(
        f"P1 converter script not found: {P1_CONVERTER_SCRIPT}\n"
        "Expected the file copied from your attached clean_csd_mofs.py."
    )

if shutil.which("git") is None:
    raise EnvironmentError("git was not found on PATH. Install git and restart the kernel.")

Python executable: c:\Users\james\CCDC\ccdc-software\csd-python-api\miniconda\python.exe
Python version: 3.11.14
Project root: c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB
Input CIF directory: C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\CIFs_to_modify\To_modify_MS
P1 converter script: c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\scripts\clean_csd_mofs.py
SAMOSA repo directory: c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\external\SAMOSA


## 0) Prepare SAMOSA and Python Dependencies

This cell clones or updates SAMOSA and checks required Python modules.

Notes:
- `ccdc` must come from the CSD Python API installation.
- If `AUTO_INSTALL_PY_DEPS=True`, missing pip-installable modules are installed automatically.

In [3]:
# Clone or update SAMOSA repository
if SAMOSA_REPO_DIR.exists():
    _ = run_cmd(["git", "-C", SAMOSA_REPO_DIR, "pull", "--ff-only"], check=False)
else:
    _ = run_cmd(["git", "clone", "https://github.com/uowoolab/SAMOSA.git", SAMOSA_REPO_DIR])

samosa_main = SAMOSA_REPO_DIR / "main.py"
if not samosa_main.exists():
    raise FileNotFoundError(f"Could not find SAMOSA main.py at: {samosa_main}")

required_modules = {
    "pandas": "pandas",
    "pymatgen": "pymatgen",
    "mendeleev": "mendeleev",
    "ccdc": "ccdc",  # comes from the licensed CSD Python API installation
}

missing = []
for module_name, pip_name in required_modules.items():
    if importlib.util.find_spec(module_name) is None:
        missing.append((module_name, pip_name))

print("\nDependency check:")
if not missing:
    print("All required modules are importable.")
else:
    for module_name, pip_name in missing:
        print(f"- Missing: {module_name} (pip name: {pip_name})")

pip_installable = [pip_name for module_name, pip_name in missing if module_name != "ccdc"]
if AUTO_INSTALL_PY_DEPS and pip_installable:
    print("\nInstalling pip dependencies:")
    _ = run_cmd([sys.executable, "-m", "pip", "install", *pip_installable], check=False)

if any(module_name == "ccdc" for module_name, _ in missing):
    print(
        "\nWARNING: 'ccdc' is missing.\n"
        "Install the CSD Python API from CCDC and ensure this kernel uses that environment."
    )

print("\nSAMOSA setup cell completed.")


$ git -C c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\external\SAMOSA pull --ff-only
Already up to date.

Dependency check:
All required modules are importable.

SAMOSA setup cell completed.


## 1) Discover Input CIF Files

This cell scans your input folder and previews the files to process.

In [4]:
input_cifs = sorted(INPUT_CIF_DIR.glob("*.cif"))
print(f"Found {len(input_cifs)} CIF files in {INPUT_CIF_DIR}")

if len(input_cifs) == 0:
    raise RuntimeError("No CIF files found in the configured input directory.")

preview_df = pd.DataFrame(
    {
        "file": [p.name for p in input_cifs],
        "size_kb": [round(p.stat().st_size / 1024, 2) for p in input_cifs],
    }
)
display(preview_df.head(20))

Found 3 CIF files in C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\CIFs_to_modify\To_modify_MS


,file,size_kb
0,CCDC__NAXLEF_hydrogens_added.cif,3.81
1,CCDC__QOSXAY_hydrogens_added.cif,5.66
2,CCDC__WOQVAB_hydrogens_added.cif,2.71


## 2) Convert All CIFs to P1 Symmetry

This runs the attached converter script (`scripts/clean_csd_mofs.py`) for each input CIF.

- If `OVERWRITE_P1=False`, existing `_P1.cif` files are skipped.
- A conversion report is saved to `data/samosa_pipeline/reports/p1_conversion_report.csv`.

In [5]:
if OVERWRITE_P1:
    for existing in P1_OUTPUT_DIR.glob("*_P1.cif"):
        existing.unlink()

conversion_rows = []
start_batch = datetime.now()

for idx, cif_path in enumerate(input_cifs, start=1):
    expected_output = P1_OUTPUT_DIR / f"{cif_path.stem}_P1.cif"

    if expected_output.exists() and not OVERWRITE_P1:
        conversion_rows.append(
            {
                "input_cif": cif_path.name,
                "output_cif": expected_output.name,
                "status": "skipped_existing",
                "returncode": 0,
                "stderr_tail": "",
            }
        )
        continue

    cmd = [
        sys.executable,
        P1_CONVERTER_SCRIPT,
        cif_path.name,
        "--read_dir",
        INPUT_CIF_DIR,
        "--write_dir",
        P1_OUTPUT_DIR,
        "-inp_is_cif",
    ]
    result = run_cmd(cmd, check=False, show_stdout=False, show_stderr=False)

    ok = result.returncode == 0 and expected_output.exists()
    conversion_rows.append(
        {
            "input_cif": cif_path.name,
            "output_cif": expected_output.name,
            "status": "ok" if ok else "failed",
            "returncode": result.returncode,
            "stderr_tail": (result.stderr or "").strip()[-300:],
        }
    )

    if not ok:
        print(f"[{idx}/{len(input_cifs)}] FAILED: {cif_path.name}")

    if idx % 10 == 0 or idx == len(input_cifs):
        print(f"Progress: {idx}/{len(input_cifs)}")

conversion_df = pd.DataFrame(conversion_rows)
p1_report_path = REPORTS_DIR / "p1_conversion_report.csv"
conversion_df.to_csv(p1_report_path, index=False)

elapsed = datetime.now() - start_batch
print(f"\nP1 conversion finished in {elapsed}.")
print(conversion_df["status"].value_counts(dropna=False))
print(f"Report written to: {p1_report_path}")

failed_df = conversion_df[conversion_df["status"] == "failed"]
if not failed_df.empty:
    print("\nFailed conversions (showing up to 20):")
    display(failed_df.head(20))
else:
    print("\nNo failed conversions detected.")


$ c:\Users\james\CCDC\ccdc-software\csd-python-api\miniconda\python.exe c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\scripts\clean_csd_mofs.py CCDC__NAXLEF_hydrogens_added.cif --read_dir C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\CIFs_to_modify\To_modify_MS --write_dir c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\data\samosa_pipeline\01_p1_cifs_MS -inp_is_cif

$ c:\Users\james\CCDC\ccdc-software\csd-python-api\miniconda\python.exe c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\scripts\clean_csd_mofs.py CCDC__QOSXAY_hydrogens_added.cif --read_dir C:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\CIFs_to_modify\To_modify_MS --write_dir c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroc

## 3) Run SAMOSA on the P1 CIF Set

This executes SAMOSA `main.py` on all `*_P1.cif` files in the conversion output folder.

Outputs are written under `data/samosa_pipeline/02_samosa_output`, including:
- `MOFs_removed_solvent/`
- `Solvent_removal_results.csv` (or `Free_solvent_removal_results.csv` if `KEEP_BOUND=True`)

In [6]:
p1_cifs = sorted(P1_OUTPUT_DIR.glob("*_P1.cif"))
print(f"P1 CIF files available for SAMOSA: {len(p1_cifs)}")

if len(p1_cifs) == 0:
    raise RuntimeError(
        "No P1 CIF files found. Run the P1 conversion cell first and resolve any failures."
    )

if OVERWRITE_SAMOSA_OUTPUT and SAMOSA_OUTPUT_DIR.exists():
    shutil.rmtree(SAMOSA_OUTPUT_DIR)
    SAMOSA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

samosa_cmd = [
    sys.executable,
    samosa_main,
    "--files_path",
    P1_OUTPUT_DIR,
    "--export_path",
    SAMOSA_OUTPUT_DIR,
    "--n_processes",
    str(N_PROCESSES),
    "--removable_denticity",
    str(REMOVABLE_DENTICITY),
    "--logging",
    LOGGING_LEVEL,
]

if VERBOSE:
    samosa_cmd.append("--verbose")
if KEEP_BOUND:
    samosa_cmd.append("--keep_bound")
if KEEP_OXO:
    samosa_cmd.append("--keep_oxo")

samosa_result = run_cmd(samosa_cmd, cwd=SAMOSA_REPO_DIR, check=False)

if samosa_result.returncode != 0:
    raise RuntimeError("SAMOSA run failed. Inspect command output above for details.")

print("\nSAMOSA run completed successfully.")

P1 CIF files available for SAMOSA: 3

$ c:\Users\james\CCDC\ccdc-software\csd-python-api\miniconda\python.exe c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\external\SAMOSA\main.py --files_path c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\data\samosa_pipeline\01_p1_cifs_MS --export_path c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\data\samosa_pipeline\02_samosa_output --n_processes 7 --removable_denticity 1 --logging INFO --verbose
No solvent or counterions identified
*********************
No solvent or counterions identified
*********************
No solvent or counterions identified
*********************
2026-03-29 13:07:44 | INFO | root (main) | Selected logging level: logging.INFO
2026-03-29 13:07:44 | INFO | root (main) | 3 cif files detected in c:\Users\james\On

## 4) Inspect Outputs and QC Flags

This section checks the generated CIFs and reads the SAMOSA summary CSV to quickly inspect solvent-removal outcomes.

In [7]:
removed_dir = SAMOSA_OUTPUT_DIR / "MOFs_removed_solvent"
result_csv_candidates = [
    SAMOSA_OUTPUT_DIR / "Solvent_removal_results.csv",
    SAMOSA_OUTPUT_DIR / "Free_solvent_removal_results.csv",
]
result_csv = next((p for p in result_csv_candidates if p.exists()), None)

removed_cifs = sorted(removed_dir.glob("*.cif")) if removed_dir.exists() else []
print(f"Removed-solvent CIF count: {len(removed_cifs)}")
print(f"Removed-solvent directory: {removed_dir}")

if result_csv is None:
    print("No SAMOSA summary CSV found yet.")
else:
    print(f"SAMOSA summary CSV: {result_csv}")
    stats_df = pd.read_csv(result_csv)
    print(f"Rows: {len(stats_df)} | Columns: {len(stats_df.columns)}")

    if "Solvent" in stats_df.columns:
        print("\nSolvent flag distribution:")
        print(stats_df["Solvent"].value_counts(dropna=False))

    flag_cols = [c for c in stats_df.columns if "flag" in c.lower()]
    if flag_cols:
        print("\nQC flag columns detected:")
        for c in flag_cols:
            print(f"- {c}")

    display(stats_df.head(30))

Removed-solvent CIF count: 21
Removed-solvent directory: c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\data\samosa_pipeline\02_samosa_output\MOFs_removed_solvent
SAMOSA summary CSV: c:\Users\james\OneDrive - Aix-Marseille Université\CNE Wroclaw 2026\CNE Thesis 2026\MOF ML Project Directory\NIST-ISODB\data\samosa_pipeline\02_samosa_output\Solvent_removal_results.csv
Rows: 46 | Columns: 22

Solvent flag distribution:
Solvent
NO     25
YES    21
Name: count, dtype: int64

QC flag columns detected:
- Atoms_match_flag
- Flag_double
- Flag_aromatic
- Metal_counterion_flag
- Terminal_oxo_flag
- Huge_counterion_flag


,CIF,Solvent,Bound_solvent,Number_of_bound molecules,Free_solvent,Number_of_free_solvent_molecules,Counterions,Number_of_counterions,Terminal_oxo,Number_of_terminal_oxo,...,Atoms_removed,Atoms_match_flag,Flag_double,Flag_aromatic,Metal_counterion_flag,Terminal_oxo_flag,Entry_terminal_oxo,Huge_counterion_flag,OH_removed,Oxo_OH
0,CELZOK01_P1.cif,NO,.,.,.,.,.,.,.,.,...,.,.,.,.,.,False,FALSE,.,.,False
1,DUXLEP_P1.cif,NO,.,.,.,.,.,.,.,.,...,.,.,.,.,.,False,FALSE,.,.,False
2,FIHKOY_P1.cif,NO,.,.,.,.,.,.,.,.,...,.,.,.,.,.,False,FALSE,.,.,False
3,GAWRED_P1.cif,NO,.,.,.,.,.,.,.,.,...,.,.,.,.,.,False,FALSE,.,.,False
4,JOFCEO01_P1.cif,NO,.,.,.,.,.,.,.,.,...,.,.,.,.,.,False,FALSE,.,.,False
5,MIDTEA_P1.cif,NO,.,.,.,.,.,.,.,.,...,.,.,.,.,.,False,FALSE,.,.,False
6,POWNOG_P1.cif,NO,.,.,.,.,.,.,.,.,...,.,.,.,.,.,False,FALSE,.,.,False
7,POWPAU_P1.cif,NO,.,.,.,.,.,.,.,.,...,.,.,.,.,.,False,FALSE,.,.,False
8,RUSCUF_P1.cif,NO,.,.,.,.,.,.,.,.,...,.,.,.,.,.,False,FALSE,.,.,False
9,SUKXUS_P1.cif,NO,.,.,.,.,.,.,.,.,...,.,.,.,.,.,False,FALSE,.,.,False


## 5) Optional Rerun Scenarios

Use these quick controls to rerun only part of the workflow:
- Set `OVERWRITE_P1=True` and rerun the P1 conversion cell to regenerate all P1 CIFs.
- Set `KEEP_BOUND=True` and rerun the SAMOSA run cell to keep bound solvents.
- Set `KEEP_OXO=True` and rerun the SAMOSA run cell to keep terminal oxo atoms.

In [ ]:
print("Notebook setup complete. Run cells top-to-bottom.")
print(f"Input CIF directory: {INPUT_CIF_DIR}")
print(f"P1 output directory: {P1_OUTPUT_DIR}")
print(f"SAMOSA output directory: {SAMOSA_OUTPUT_DIR}")
print(f"Reports directory: {REPORTS_DIR}")